# Improvement 01: MER-only checkpointing on ONE real CT case

Enable **T4 GPU**, then **Run all**. Upload the same **A0025_CT_GT.zip** when
prompted. Defaults are unchanged: one fold-0 TRAIN volume, masters 30 and 31,
two optimizer attempts, batch 1, native 11x512x512, AMP, full allocator budget.

The ONLY experimental difference is checkpoint scope: keep checkpointing in
MER query blocks, run EPA/TEP/encoder1/final decoder normally. This does not
remove any model module or change parameters, attention, loss or input values.
The validated previous helpers are reused unchanged; one small candidate
helper selects the new execution mode only within its isolated worker.

Numerical/gradient/normalization-buffer comparisons run before the real pilot
using this candidate mode. A failed comparison prevents training. OOM or FAIL
is recorded without falling back or reducing the input/model. Reports compare
with the previous A-0025 result; two updates are not a robust speed benchmark.

No full cohort, multiple improvements, production integration or full epoch
is run. Source comes from the pinned public commit plus helpers embedded below;
no GitHub push is needed for this notebook experiment.

In [ ]:
REPOSITORY = "https://github.com/torusama/merit-covid-ct.git"
SOURCE_COMMIT = "e3493dd9f2dac704d2def6225d76f19df3262ec4"
TRAIN_STEPS = 2
CUDA_MEMORY_FRACTION = 1.0

CASE_SPEC = {'case_id': 'volume-covid19-A-0025', 'group_id': 'volume-covid19-A-0025', 'fold': 0, 'ct_name': 'volume-covid19-A-0025_ct.nii.gz', 'gt_name': 'volume-covid19-A-0025_seg.nii.gz', 'shape_xyz': [512, 512, 61], 'sha256': {'ct': '105b590753619c608e241001aee1ff5c1add4f432103930825d460bf8a2bb8b6', 'gt': 'bd90a1e4a856269b8e2fcc9da18f6eecbc834750db0ba0806796692bc7d4407c'}, 'file_sizes': {'ct': 16807949, 'gt': 66366}, 'split_sha256_lf': '7b31f9e250e64826f1fff0d9fec4a4aba034252cec7d5123d5f0393ec665e54d', 'manifest_fingerprint': 'c1a313cd1e57a8e526108fd413ed052d00afab2b72ac9392595e603a742ffde0'}

## 1. Check the assigned GPU
This short-lived child process releases its CUDA context on exit. The notebook
kernel itself does not import Torch or hold model/optimizer tensors.

In [ ]:
import json
import subprocess
import sys
from importlib import metadata
from packaging.specifiers import SpecifierSet

def runtime_probe():
    source = """import json, torch
available = torch.cuda.is_available()
print(json.dumps({
    'torch': torch.__version__, 'cuda': torch.version.cuda, 'cuda_available': available,
    'gpus': [{'name': torch.cuda.get_device_name(i),
              'vram_mib': torch.cuda.get_device_properties(i).total_memory / 2**20}
             for i in range(torch.cuda.device_count())]}))
"""
    result = subprocess.run([sys.executable, "-c", source], capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError("The runtime's PyTorch could not load. Select a current GPU runtime.\n" + result.stderr)
    info = json.loads(result.stdout)
    if not info["cuda_available"]:
        raise RuntimeError("No CUDA GPU assigned. Enable a GPU in the notebook settings, reconnect, and Run all.")
    if not SpecifierSet(">=2.4.1").contains(info["torch"], prereleases=True):
        raise RuntimeError("This project needs Torch >=2.4.1. Use a newer runtime; this notebook will not replace Torch/CUDA.")
    return info

ENVIRONMENT = runtime_probe()
print(json.dumps(ENVIRONMENT, indent=2))

## 2. Fetch code and reuse installed packages
Code is cached inside the current runtime. No dataset is downloaded or inspected.
Torch and NumPy versions are constrained to the versions already supplied by the
platform; the setup cannot silently replace them. All model runs use fresh child
processes, so installing a missing library does not require restarting the kernel.

In [ ]:
from datetime import datetime, timezone
import os
from pathlib import Path
import uuid

BASE = (Path(os.environ["MERIT_CLOUD_HOME"]) if os.environ.get("MERIT_CLOUD_HOME") else
        Path("/kaggle/working") if Path("/kaggle/working").is_dir() else
        Path("/content") if Path("/content").is_dir() else Path.cwd() / "outputs" / "cloud_quickstart")
BASE.mkdir(parents=True, exist_ok=True)
PROJECT = BASE / ("merit-source-" + SOURCE_COMMIT[:12])
RUN = BASE / "merit-results" / (datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8])
RUN.mkdir(parents=True, exist_ok=False)

def git(*args, check=True):
    return subprocess.run(["git", "-C", str(PROJECT), *args], check=check,
                          capture_output=True, text=True)

if not PROJECT.exists():
    subprocess.run(["git", "clone", "--quiet", "--depth", "1", REPOSITORY, str(PROJECT)], check=True)
elif not (PROJECT / ".git").is_dir():
    raise RuntimeError(f"Source path exists but is not a Git checkout: {PROJECT}")
if git("status", "--porcelain", "--untracked-files=no").stdout.strip():
    raise RuntimeError("The cached source has local edits. Save those edits and select a fresh source directory.")
if git("cat-file", "-e", SOURCE_COMMIT + "^{commit}", check=False).returncode:
    git("fetch", "--quiet", "--depth", "1", "origin", SOURCE_COMMIT)
git("checkout", "--quiet", "--detach", SOURCE_COMMIT)
assert git("rev-parse", "HEAD").stdout.strip() == SOURCE_COMMIT
assert (PROJECT / "scripts/merit_preflight.py").is_file()

requirements = {
    "monai": "==1.5.2", "einops": "==0.8.2", "numpy": ">=1.24,<3",
    "nibabel": ">=5", "scipy": ">=1.10", "PyYAML": ">=6",
}
missing = []
for name, bounds in requirements.items():
    try:
        installed = metadata.version(name)
    except metadata.PackageNotFoundError:
        installed = None
    if installed is None or not SpecifierSet(bounds).contains(installed, prereleases=True):
        missing.append(name + bounds)

protected = {name: metadata.version(name) for name in ("torch", "numpy")}
constraints = RUN / "runtime_constraints.txt"
constraints.write_text("".join(name + "==" + version + "\n" for name, version in protected.items()))
if missing:
    print("Installing only:", ", ".join(missing), flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
                    "--constraint", str(constraints), *missing], check=True)
else:
    print("All project dependencies already satisfy the requirements; skipping pip.")
for name, version in protected.items():
    if metadata.version(name) != version:
        raise RuntimeError(f"Protected runtime package changed unexpectedly: {name}")
ENVIRONMENT["packages"] = {name: metadata.version(name) for name in requirements}
ENVIRONMENT["source_commit"] = SOURCE_COMMIT
ENVIRONMENT["source_repository"] = REPOSITORY
(RUN / "environment.json").write_text(json.dumps(ENVIRONMENT, indent=2) + "\n")
print("Source:", PROJECT)
print("Reports:", RUN)

## 3. Install the isolated experiment helpers
These files exist only in this run's report directory. The model's layer,
parameter and state shapes are unchanged. Output/gradient/buffer comparisons
must pass before the native real-data optimizer attempts start.

In [ ]:
TRIAL_SOURCE = r'''"""Experimental activation checkpointing; production model source is unchanged.

Run in a fresh process. The cloud notebook embeds this file so unpublished
runtime experiments can be tested against its pinned public model source.
"""

import argparse
from copy import deepcopy
import gc
import json
import os
from pathlib import Path
import sys
import types

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))

import torch
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint

from merit_covid.models.merit_unetrpp import MERITUNETRPP, MERITLoss, MasterEvidenceRouting
from merit_covid.models.unetrpp.epa import EPA
from merit_covid.models.merit_unetrpp.modules import (
    _check_features, master_positions, relative_offsets, sinusoidal_encoding,
)


def checkpointed_mer(self, features, master_index, z_offsets_mm=None):
    """Original MER equations; recompute distance/bias/attention per query block.

    The block still attends ALL context keys in one softmax. Large distances
    must be constructed inside the checkpoint, not saved as checkpoint inputs.
    Loop-dependent coordinates are explicit inputs, not late-bound closures.
    """
    _check_features(features, self.channels)
    batch, channels, depth, height, width = features.shape
    if depth < 2:
        raise ValueError("MER needs at least one context slice")
    master = master_positions(master_index, batch, depth, features.device)
    rows = torch.arange(batch, device=features.device)
    positions = torch.arange(depth, device=features.device).expand(batch, -1)
    context_indices = positions[positions != master[:, None]].reshape(batch, depth - 1)
    slices = features.permute(0, 2, 3, 4, 1).reshape(batch, depth, height * width, channels)
    master_features = slices[rows, master]
    context = slices[rows[:, None], context_indices]
    offsets = relative_offsets(master, depth, mode=self.position_mode,
        z_offsets_mm=z_offsets_mm, unit_mm=self.position_unit_mm)[rows[:, None], context_indices]
    pe = sinusoidal_encoding(offsets, channels, dtype=context.dtype)[:, :, None]
    q = self.query(master_features)
    k = self.key(context + self.position_scale * pe).flatten(1, 2)
    v = self.value(context).flatten(1, 2)
    coords = torch.stack(torch.meshgrid(torch.arange(height, device=features.device),
        torch.arange(width, device=features.device), indexing="ij"), -1).reshape(-1, 2)

    def attention(query, keys, values, query_coords, all_coords, scale_raw):
        delta = (query_coords[:, None] - all_coords[None]).square().sum(-1)
        distance = (delta / float(height ** 2 + width ** 2)).repeat(1, depth - 1)
        bias = -F.softplus(scale_raw) * distance
        return F.scaled_dot_product_attention(query, keys, values, attn_mask=bias, dropout_p=0.0)

    updates = []
    for start in range(0, height * width, self.query_chunk_size):
        stop = start + self.query_chunk_size
        args = (q[:, start:stop], k, v, coords[start:stop], coords, self.distance_scale_raw)
        update = (checkpoint(attention, *args, use_reentrant=False, preserve_rng_state=True)
                  if self.training and torch.is_grad_enabled() else attention(*args))
        updates.append(update)
    evidence = self.output(torch.cat(updates, dim=1))
    gate = self.gate(torch.cat((master_features, evidence), -1))
    updated = master_features + self.beta * gate * evidence
    result = slices.clone()
    result[rows, master] = updated
    return result.reshape(batch, depth, height, width, channels).permute(0, 4, 1, 2, 3).contiguous()


def enable_checkpointing(model):
    """Patch execution only; keep parameters, state keys and model config intact."""
    if getattr(model, "_memory_trial_enabled", False):
        return model
    before = {name: id(value) for name, value in model.state_dict(keep_vars=True).items()}
    if model.mer is not None:
        model.mer.forward = types.MethodType(checkpointed_mer, model.mer)

    def wrap(module):
        if any(getattr(child, "track_running_stats", False) for child in module.modules()):
            raise ValueError("Checkpoint region contains mutable normalization statistics")
        original = module.forward
        def forward(*args, **kwargs):
            if not module.training or not torch.is_grad_enabled():
                return original(*args, **kwargs)
            return checkpoint(original, *args, use_reentrant=False, preserve_rng_state=True, **kwargs)
        module.forward = forward

    # Transformer conv51 contains BatchNorm even with norm_name='instance'.
    # Checkpoint only its stateless EPA, leaving BatchNorm and its updates alone.
    regions = [module for module in model.modules() if isinstance(module, EPA)]
    regions += [model.encoder1, model.decoder2]
    if model.tep is not None:
        regions.append(model.tep)
    for module in regions:
        wrap(module)
    after = {name: id(value) for name, value in model.state_dict(keep_vars=True).items()}
    if before != after:
        raise AssertionError("Checkpoint execution patch changed model state identities")
    model._memory_trial_enabled = True
    return model


def _compare_grads(reference, trial, *, rtol, atol):
    errors = []
    for (name, left), (other_name, right) in zip(reference.named_parameters(), trial.named_parameters()):
        assert name == other_name
        if left.grad is None or right.grad is None:
            raise AssertionError(f"Missing parameter gradient: {name}")
        torch.testing.assert_close(left.grad, right.grad, rtol=rtol, atol=atol, msg=name)
        if not torch.isfinite(right.grad).all():
            raise AssertionError(f"Nonfinite parameter gradient: {name}")
        errors.append(float((left.grad - right.grad).abs().max()))
    return max(errors, default=0.)


def mer_equivalence():
    torch.manual_seed(123)
    reference = MasterEvidenceRouting(4, attention_dim=8, gate_hidden=3, query_chunk_size=5).double().train()
    trial = deepcopy(reference)
    trial.forward = types.MethodType(checkpointed_mer, trial)
    x = torch.randn(2, 4, 3, 4, 6, dtype=torch.float64, requires_grad=True)
    y = x.detach().clone().requires_grad_()
    expected = reference(x, [0, 2])
    actual = trial(y, [0, 2])
    torch.testing.assert_close(expected, actual, rtol=1e-10, atol=1e-12)
    expected.square().mean().backward()
    actual.square().mean().backward()
    torch.testing.assert_close(x.grad, y.grad, rtol=1e-10, atol=1e-12)
    error = _compare_grads(reference, trial, rtol=1e-10, atol=1e-12)
    return {"status": "PASS", "dtype": "float64", "off_center_masters": True,
            "max_parameter_gradient_abs_error": error}


def model_equivalence(device, amp=False):
    torch.manual_seed(321)
    reference = MERITUNETRPP(img_size=(11, 32, 32), feature_size=2, depths=(1, 1, 1, 1)).to(device).train()
    trial = enable_checkpointing(deepcopy(reference))
    assert list(reference.state_dict()) == list(trial.state_dict())
    assert sum(p.numel() for p in reference.parameters()) == sum(p.numel() for p in trial.parameters())
    x = torch.randn(1, 1, 11, 32, 32, device=device, requires_grad=True)
    target = torch.zeros_like(x)
    target[:, :, 4:7, 5:9, 7:12] = 1
    criterion = MERITLoss()
    rng = torch.get_rng_state()
    cuda_rng = torch.cuda.get_rng_state_all() if device.type == "cuda" else None
    def evaluate(model, image):
        torch.set_rng_state(rng)
        if cuda_rng is not None:
            torch.cuda.set_rng_state_all(cuda_rng)
        with torch.autocast(device.type, dtype=torch.float16, enabled=amp):
            output = model(image, [5])
            losses = criterion(output, target, [5])
        losses["total"].backward()
        return output, losses
    y = x.detach().clone().requires_grad_()
    expected, expected_loss = evaluate(reference, x)
    actual, actual_loss = evaluate(trial, y)
    rtol, atol = (1e-3, 1e-5) if amp else (1e-4, 1e-6)
    for left, right in zip(expected, actual):
        torch.testing.assert_close(left, right, rtol=rtol, atol=atol)
    for name in expected_loss:
        torch.testing.assert_close(expected_loss[name], actual_loss[name], rtol=rtol, atol=atol)
    torch.testing.assert_close(x.grad, y.grad, rtol=rtol, atol=atol)
    error = _compare_grads(reference, trial, rtol=rtol, atol=atol)
    for name, value in reference.named_buffers():
        torch.testing.assert_close(value, dict(trial.named_buffers())[name], rtol=0, atol=0, msg=name)
    return {"status": "PASS", "amp": amp, "device": str(device), "rtol": rtol, "atol": atol,
            "same_parameter_count_and_state_keys": True, "dropout_rng_preserved": True,
            "normalization_buffers_unchanged": True,
            "max_output_abs_error": max(float((a - b).abs().max().detach()) for a, b in zip(expected, actual)),
            "max_input_gradient_abs_error": float((x.grad - y.grad).abs().max()),
            "max_parameter_gradient_abs_error": error}


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", required=True)
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument("--size", type=int, default=512)
    parser.add_argument("--memory-fraction", type=float, default=1.)
    parser.add_argument("--validate-only", action="store_true")
    args = parser.parse_args()
    if not 0 < args.memory_fraction <= 1:
        parser.error("memory fraction must be in (0,1]")
    args.output.parent.mkdir(parents=True, exist_ok=True)
    if args.output.exists():
        raise FileExistsError(args.output)
    torch.set_num_threads(2)
    torch.use_deterministic_algorithms(True)
    device = torch.device("cuda:0")
    torch.cuda.set_per_process_memory_fraction(args.memory_fraction, 0)
    validation = {"status": "FAIL"}
    path = args.output.with_name(args.output.stem + "_equivalence.json")
    if path.exists():
        raise FileExistsError(path)
    try:
        validation["mer"] = mer_equivalence()
        validation["model_fp32"] = model_equivalence(device)
        validation["model_amp"] = model_equivalence(device, amp=True)
        validation["status"] = "PASS"
    except Exception as error:
        validation["error"] = str(error)
        raise
    finally:
        path.write_text(json.dumps(validation, indent=2) + "\n", encoding="utf-8")
    print("Output/gradient equivalence: PASS", flush=True)
    if args.validate_only:
        return 0
    gc.collect()
    torch.cuda.empty_cache()
    from merit_covid.engine import preflight
    original_class = preflight.MERITUNETRPP
    def factory(**kwargs):
        return enable_checkpointing(original_class(**kwargs))
    preflight.MERITUNETRPP = factory
    report = preflight.profile_model(args.config, args.output, size=args.size, batch_size=1,
        device="cuda", warmup=1, repeats=1)
    report.update(checkpoint_trial=True, equivalence=validation,
        cloud_allocator_fraction=args.memory_fraction, production_model_source_modified=False,
        execution_regions="MER query blocks including bias; EPA, TEP, encoder1 and final decoder; BatchNorm excluded")
    args.output.write_text(json.dumps(report, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    print("Checkpoint trial:", report["status"], flush=True)
    return 0 if report["status"] == "PASS" else 1


if __name__ == "__main__":
    raise SystemExit(main())
'''

import hashlib
TRIAL_FILE = RUN / "merit_checkpoint_trial.py"
TRIAL_FILE.write_text(TRIAL_SOURCE, encoding="utf-8")
ENVIRONMENT["checkpoint_trial_sha256"] = hashlib.sha256(TRIAL_SOURCE.encode("utf-8")).hexdigest()
ENVIRONMENT["checkpoint_trial"] = True
(RUN / "environment.json").write_text(json.dumps(ENVIRONMENT, indent=2) + "\n")
print("Experimental helper SHA-256:", ENVIRONMENT["checkpoint_trial_sha256"])


In [ ]:
REAL_TRIAL_SOURCE = r'''"""Bounded native CT/GT pilot using shared Dataset/model/loss, not full training.

Only the uploaded, hash-verified TRAIN case is opened. The canonical split is
checked without fabricating paths/files for the missing 198 cases. No patient
data or large model checkpoint is copied into the report directory.
"""

import argparse
import gc
import hashlib
import json
from pathlib import Path
import time
import traceback
from unittest.mock import patch

import nibabel as nib
import numpy as np
import torch

if __package__:
    from .merit_checkpoint_trial import enable_checkpointing, mer_equivalence, model_equivalence
else:
    from merit_checkpoint_trial import enable_checkpointing, mer_equivalence, model_equivalence

from merit_covid.data.pipeline.cases import Case
from merit_covid.data.pipeline.config import PreprocessConfig
from merit_covid.data.pipeline.dataset import CTSlabDataset
from merit_covid.data.pipeline.dataloader import make_dataloader
from merit_covid.data.pipeline.geometry import reorient_to_lps
from merit_covid.engine.training import load_experiment, validate_experiment
from merit_covid.engine.preflight import _setup, _sync, _memory
from merit_covid.models.merit_unetrpp import MERITUNETRPP, MERITUNETRPPAdapter, MERITLoss


def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def parameter_hash(model):
    digest = hashlib.sha256()
    for name, value in model.named_parameters():
        digest.update(name.encode())
        digest.update(value.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()


def verified_case(ct, mask, split_path, spec):
    ct, mask, split_path = Path(ct).resolve(), Path(mask).resolve(), Path(split_path)
    if ct == mask:
        raise ValueError("CT and mask must be distinct source files")
    raw = split_path.read_bytes().replace(b"\r\n", b"\n")
    if hashlib.sha256(raw).hexdigest() != spec["split_sha256_lf"]:
        raise ValueError("Canonical split bytes do not match the frozen specification")
    split = json.loads(raw)
    if split["manifest_fingerprint"] != spec["manifest_fingerprint"]:
        raise ValueError("Canonical manifest identity mismatch")
    part = next(row for row in split["folds"] if row["fold"] == spec["fold"])
    if (spec["case_id"] not in part["train"]["case_ids"]
            or spec["group_id"] not in part["train"]["group_ids"]
            or any(spec["case_id"] in part[key]["case_ids"] or spec["group_id"] in part[key]["group_ids"]
                   for key in ("validation", "test"))):
        raise ValueError("Uploaded case/group must belong only to the selected TRAIN partition")
    hashes = {"ct": file_hash(ct), "gt": file_hash(mask)}
    if hashes != spec["sha256"]:
        raise ValueError(f"Uploaded CT/GT bytes do not match the expected case: {hashes}")
    for source in (ct, mask):
        if list(nib.load(source).shape) != spec["shape_xyz"]:
            raise ValueError("Source shape does not match the native case specification")
    return Case(spec["case_id"], ct, mask, spec["group_id"]), hashes, split["identity"]


def verify_sample(sample, ct, mask):
    indices = sample["slice_indices"].tolist()
    image = np.stack([np.asarray(ct.dataobj[:, :, i], dtype=np.float32).T for i in indices])
    truth = np.stack([np.asarray(mask.dataobj[:, :, i], dtype=np.float32).T for i in indices])
    np.testing.assert_array_equal(sample["image_slab"][0].numpy(), image)
    np.testing.assert_array_equal(sample["mask_slab"][0].numpy(), truth)
    info = sample["metadata"]
    if (info["lesion_retention"]["resize_applied"] or info["z_resampled"]
            or info["processed_size_hw"] != info["native_size_hw"]):
        raise AssertionError("Native input was resized or resampled")
    return {"master": int(sample["master_slice_index"]), "selected_slices": indices,
            "ct_gt_equal_native_source": True,
            "positive_voxels_per_selected_slice": (truth == 1).sum((1, 2)).tolist(),
            "presence_per_selected_slice": (truth == 1).any((1, 2)).tolist(),
            "ct_range": [float(image.min()), float(image.max())]}


def run_trial(config_path, ct, mask, split_path, spec, output, *, steps=2,
              device="cuda", memory_fraction=1., check_equivalence=True):
    output = Path(output)
    if output.exists():
        raise FileExistsError(output)
    output.parent.mkdir(parents=True, exist_ok=True)
    report = {"status": "NOT_RUN", "scope": "one verified TRAIN case, bounded native optimizer attempts",
              "requested_attempts": steps, "optimizer_steps": 0, "amp_skipped_steps": 0,
              "full_training": False, "accuracy_evaluated": False, "checkpoint_saved": False,
              "production_source_modified": False, "input_checks": [], "batches": []}
    pilot_start = time.perf_counter()
    resolved, case, before = None, None, None
    try:
        if type(steps) is not int or not 1 <= steps <= 8:
            raise ValueError("Pilot attempts must be between 1 and 8")
        if not 0 < memory_fraction <= 1:
            raise ValueError("memory_fraction must be in (0,1]")
        case, before, identity = verified_case(ct, mask, split_path, spec)
        report.update(case_id=case.case_id, fold=spec["fold"], identity=identity,
            source_sha256_before=before, split_sha256_lf=spec["split_sha256_lf"])
        config = load_experiment(config_path)
        if config["preprocess"]["intensity_encoding"] != "native" or config["preprocess"]["hu_window"] is not None:
            raise ValueError("This pilot requires native intensity without a HU window")
        if config["preprocess"]["sampling_mode"] != "index" or config["preprocess"]["slab_size"] != 11:
            raise ValueError("This pilot requires the existing 11-slice index sampler")
        source_ct, source_gt = reorient_to_lps(nib.load(case.image)), reorient_to_lps(nib.load(case.mask))
        native_hw = [source_ct.shape[1], source_ct.shape[0]]
        config["preprocess"]["size_hw"] = native_hw
        config["model"]["img_size"] = [11, *native_hw]
        config["training"].update(device=device, batch_size=1, num_workers=0)
        config = validate_experiment(config)
        resolved = _setup(config)
        if resolved.type == "cuda":
            torch.cuda.set_per_process_memory_fraction(float(memory_fraction),
                resolved.index if resolved.index is not None else torch.cuda.current_device())
        report.update(config=config, torch_version=torch.__version__, cuda_runtime=torch.version.cuda,
            hardware=torch.cuda.get_device_name(resolved) if resolved.type == "cuda" else "CPU",
            native_shape_xyz=list(source_ct.shape), native_size_hw=native_hw,
            cloud_allocator_fraction=memory_fraction)
        input_start = time.perf_counter()
        dataset = CTSlabDataset([case], PreprocessConfig(**config["preprocess"]))
        masters = [(len(dataset) // 2 + i) % len(dataset) for i in range(steps)]
        def forbidden_resize(*args, **kwargs):
            raise AssertionError("Native pilot attempted XY interpolation")
        with patch("merit_covid.data.pipeline.preprocessing.resize_in_plane", forbidden_resize):
            for master in masters:
                sample = dataset[master]
                report["input_checks"].append(verify_sample(sample, source_ct, source_gt))
                if "geometry" not in report:
                    report["geometry"] = {key: sample["metadata"][key] for key in (
                        "original_orientation", "canonical_orientation", "original_shape_xyz", "canonical_shape_xyz",
                        "original_affine_mm", "canonical_affine_mm", "spatial_unit_in_header", "assume_mm",
                        "nifti_scaling", "intensity_encoding", "hu_window") if key in sample["metadata"]}
        report.update(master_indices=masters, input_shape=[1, 1, 11, *native_hw],
            xy_interpolation_called=False, normalization_applied=False,
            dataset_and_native_comparison_seconds=time.perf_counter() - input_start)
        if check_equivalence:
            report["equivalence"] = {"mer": mer_equivalence(), "fp32": model_equivalence(resolved)}
            if resolved.type == "cuda":
                report["equivalence"]["amp"] = model_equivalence(resolved, amp=True)
            gc.collect()
            if resolved.type == "cuda":
                torch.cuda.empty_cache()
        _setup(config)  # Restore the experiment seed after the independent comparisons.
        if resolved.type == "cuda":
            torch.cuda.reset_peak_memory_stats(resolved)
        settings = config["training"]
        model = enable_checkpointing(MERITUNETRPP(**config["model"]).to(resolved)).train()
        model.set_epoch(0, warmup_epochs=settings["warmup_epochs"])
        report.update(parameter_count=sum(p.numel() for p in model.parameters()),
                      actual_pcg_ramp=float(model.pcg_ramp))
        criterion = MERITLoss(**config["loss"])
        optimizer = torch.optim.AdamW(model.parameters(), lr=settings["learning_rate"], weight_decay=settings["weight_decay"])
        amp = settings["amp"] and resolved.type == "cuda"
        scaler = torch.amp.GradScaler("cuda", enabled=amp, init_scale=settings["amp_init_scale"])
        loader = make_dataloader(torch.utils.data.Subset(dataset, masters), batch_size=1,
            shuffle=False, num_workers=0, seed=settings["seed"])
        report["parameter_sha256_before"] = parameter_hash(model)
        with patch("merit_covid.data.pipeline.preprocessing.resize_in_plane", forbidden_resize):
            for i, batch in enumerate(loader):
                optimizer.zero_grad(set_to_none=True)
                _sync(resolved)
                start = time.perf_counter()
                image, truth = batch["image_slab"].to(resolved), batch["mask_slab"].to(resolved)
                with torch.autocast(resolved.type, dtype=torch.float16, enabled=amp):
                    prediction = model(image, batch["master_index"], batch["z_offsets_mm"])
                    losses = criterion(prediction, truth, batch["master_index"],
                        valid_region=batch["valid_region"].to(resolved), z_offsets_mm=batch["z_offsets_mm"])
                if not all(torch.isfinite(value).all() for value in losses.values()):
                    raise FloatingPointError("Nonfinite real-CT loss before optimizer update")
                scaler.scale(losses["total"]).backward()
                if any(p.grad is None for p in model.parameters()):
                    raise FloatingPointError("Missing real-CT parameter gradients")
                if not amp and any(not torch.isfinite(p.grad).all() for p in model.parameters()):
                    raise FloatingPointError("Nonfinite real-CT gradients")
                scaler.unscale_(optimizer)
                norm = torch.nn.utils.clip_grad_norm_(model.parameters(), settings["gradient_clip"], error_if_nonfinite=not amp)
                previous = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                _sync(resolved)
                skipped = scaler.get_scale() < previous
                report["optimizer_steps"] += int(not skipped)
                report["amp_skipped_steps"] += int(skipped)
                row = {"attempt": i + 1, "master": masters[i], "seconds": time.perf_counter() - start,
                       "skipped": skipped, "gradient_norm_finite": bool(torch.isfinite(norm)),
                       "losses": {key: float(value.detach()) for key, value in losses.items()}}
                report["batches"].append(row)
                if not skipped and not row["gradient_norm_finite"]:
                    raise FloatingPointError("Effective optimizer attempt had a nonfinite gradient norm")
                print(f"Real CT attempt {i+1}/{steps}: master={masters[i]}, seconds={row['seconds']:.3f}, skipped={skipped}", flush=True)
                del prediction, losses, image, truth
        report["parameter_sha256_after"] = parameter_hash(model)
        report["weights_changed"] = report["parameter_sha256_before"] != report["parameter_sha256_after"]
        if report["optimizer_steps"] != steps or not report["weights_changed"]:
            raise FloatingPointError("Requested effective optimizer updates were not all completed")
        optimizer.zero_grad(set_to_none=True)
        # One CT-only master prediction, not an incomplete full-volume export.
        model.eval()
        original_load = nib.load
        def ct_only_load(filename, *args, **kwargs):
            if Path(filename).resolve() == case.mask.resolve():
                raise AssertionError("CT-only inference attempted to open GT")
            return original_load(filename, *args, **kwargs)
        with patch.object(nib, "load", ct_only_load), torch.inference_mode():
            inference = CTSlabDataset([case], dataset.config, require_mask=False)
            batch = next(iter(make_dataloader(torch.utils.data.Subset(inference, [masters[0]]))))
            batch["image_slab"] = batch["image_slab"].to(resolved)
            logits = MERITUNETRPPAdapter(model).predict_logits(batch)
            if logits.shape != (1, 1, *native_hw) or not torch.isfinite(logits).all():
                raise FloatingPointError("Invalid CT-only native master logits")
            probability = logits.float().sigmoid()
            report["inference"] = {"ct_only_guard": "PASS", "master_only": True,
                "full_volume_reconstructed": False, "shape": list(logits.shape),
                "probability_range": [float(probability.min()), float(probability.max())]}
        report["status"] = "PASS"
    except Exception as error:
        report.update(status="OOM" if isinstance(error, torch.OutOfMemoryError) else "FAIL",
            error_type=type(error).__name__, error=str(error), traceback=traceback.format_exc())
    finally:
        if case is not None and before is not None:
            after = {"ct": file_hash(case.image), "gt": file_hash(case.mask)}
            report.update(source_sha256_after=after, source_files_unchanged=after == before)
            if after != before:
                report.update(status="FAIL", error="Source files changed during pilot")
        if resolved is not None:
            report.update(_memory(resolved))
        report["total_pilot_seconds"] = time.perf_counter() - pilot_start
        output.write_text(json.dumps(report, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    return report


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    for name in ("config", "ct", "mask", "split", "spec", "output"):
        parser.add_argument("--" + name, type=Path, required=True)
    parser.add_argument("--steps", type=int, default=2)
    parser.add_argument("--memory-fraction", type=float, default=1.)
    args = parser.parse_args()
    report = run_trial(args.config, args.ct, args.mask, args.split,
        json.loads(args.spec.read_text()), args.output, steps=args.steps, memory_fraction=args.memory_fraction)
    print("Real CT pilot:", report["status"], flush=True)
    return 0 if report["status"] == "PASS" else 1


if __name__ == "__main__":
    raise SystemExit(main())
'''

REAL_TRIAL_FILE = RUN / "merit_real_case_trial.py"
REAL_TRIAL_FILE.write_text(REAL_TRIAL_SOURCE, encoding="utf-8")
SPEC_FILE = RUN / "case_specification.json"
SPEC_FILE.write_text(json.dumps(CASE_SPEC, indent=2) + "\n")
ENVIRONMENT["real_case_trial_sha256"] = hashlib.sha256(REAL_TRIAL_SOURCE.encode()).hexdigest()
ENVIRONMENT["case_id"] = CASE_SPEC["case_id"]
(RUN / "environment.json").write_text(json.dumps(ENVIRONMENT, indent=2) + "\n")

In [ ]:
CANDIDATE_SOURCE = r'''"""One-case experiment: checkpoint MER query blocks only, retaining the model.

The previously validated checkpoint and real-case helpers are reused unchanged.
Apply the candidate only to fresh model instances in an isolated worker.
"""

import argparse
from contextlib import contextmanager
import json
from pathlib import Path
import types

if __package__:
    from . import merit_checkpoint_trial as memory
    from . import merit_real_case_trial as pilot
else:
    import merit_checkpoint_trial as memory
    import merit_real_case_trial as pilot


def enable_mer_only(model):
    if getattr(model, "_checkpoint_trial_mode", None) == "mer_only":
        return model
    if getattr(model, "_memory_trial_enabled", False):
        raise ValueError("MER-only mode needs a fresh model, not an already patched baseline")
    if model.mer is None:
        raise ValueError("MER-only trial requires MER enabled")
    before = {name: id(value) for name, value in model.state_dict(keep_vars=True).items()}
    model.mer.forward = types.MethodType(memory.checkpointed_mer, model.mer)
    after = {name: id(value) for name, value in model.state_dict(keep_vars=True).items()}
    if before != after:
        raise AssertionError("Execution patch changed model parameter/buffer identities")
    model._checkpoint_trial_mode = "mer_only"
    model._memory_trial_enabled = True
    return model


@contextmanager
def candidate_execution():
    """Gate both numerical comparisons and the pilot on the candidate mode."""
    old_memory, old_pilot = memory.enable_checkpointing, pilot.enable_checkpointing
    memory.enable_checkpointing = pilot.enable_checkpointing = enable_mer_only
    try:
        yield
    finally:
        memory.enable_checkpointing, pilot.enable_checkpointing = old_memory, old_pilot


def run_candidate(*args, **kwargs):
    with candidate_execution():
        report = pilot.run_trial(*args, **kwargs)
    report.update(execution_mode="mer_only", checkpointed_regions=["MER query blocks including distance bias"],
        uncheckpointed_regions=["EPA", "TEP", "encoder1", "final decoder"],
        comparison_scope="one case, two fixed masters; not a cohort speed benchmark")
    output = Path(args[5] if len(args) > 5 else kwargs["output"])
    output.write_text(json.dumps(report, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    return report


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    for name in ("config", "ct", "mask", "split", "spec", "output"):
        parser.add_argument("--" + name, type=Path, required=True)
    parser.add_argument("--steps", type=int, default=2)
    parser.add_argument("--memory-fraction", type=float, default=1.)
    args = parser.parse_args()
    report = run_candidate(args.config, args.ct, args.mask, args.split,
        json.loads(args.spec.read_text()), args.output, steps=args.steps, memory_fraction=args.memory_fraction)
    print("MER-only real pilot:", report["status"], flush=True)
    return 0 if report["status"] == "PASS" else 1


if __name__ == "__main__":
    raise SystemExit(main())
'''

CANDIDATE_FILE = RUN / "merit_mer_only_trial.py"
CANDIDATE_FILE.write_text(CANDIDATE_SOURCE, encoding="utf-8")
ENVIRONMENT["mer_only_trial_sha256"] = hashlib.sha256(CANDIDATE_SOURCE.encode()).hexdigest()
ENVIRONMENT["execution_mode"] = "mer_only"
(RUN / "environment.json").write_text(json.dumps(ENVIRONMENT, indent=2) + "\n")

## 4. Upload the real CT and GT
Choose `A0025_CT_GT.zip`, or select BOTH
`volume-covid19-A-0025_ct.nii.gz` and `volume-covid19-A-0025_seg.nii.gz`.
Correct inputs are reused on subsequent runs in the same runtime. Files are
stored outside the report folder, so patient images are not put into the report
ZIP. A wrong or incomplete upload produces a failure report, not a model run.

In [ ]:
import tempfile
import zipfile
import shutil

def input_hash(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def inputs_ok(directory):
    try:
        return (input_hash(directory / CASE_SPEC["ct_name"]) == CASE_SPEC["sha256"]["ct"] and
                input_hash(directory / CASE_SPEC["gt_name"]) == CASE_SPEC["sha256"]["gt"])
    except (OSError, TypeError):
        return False

UPLOAD_ERROR = None
cache = BASE / "merit-real-input-A0025.json"
INPUT_DIR = None
if cache.is_file():
    try:
        candidate = Path(json.loads(cache.read_text())["directory"])
        if inputs_ok(candidate):
            INPUT_DIR = candidate
    except (ValueError, OSError, KeyError):
        pass
if INPUT_DIR is None:
    parent = BASE / "merit-real-inputs"
    parent.mkdir(exist_ok=True)
    INPUT_DIR = Path(tempfile.mkdtemp(prefix="A0025_", dir=parent))
    try:
        from google.colab import files
        previous = Path.cwd()
        try:
            os.chdir(INPUT_DIR)
            print("Select A0025_CT_GT.zip, or BOTH CT and mask .nii.gz files.", flush=True)
            uploaded = files.upload()
            del uploaded
        finally:
            os.chdir(previous)
        if not inputs_ok(INPUT_DIR):
            archives = list(INPUT_DIR.glob("*.zip"))
            if len(archives) == 1:
                with zipfile.ZipFile(archives[0]) as archive:
                    names = [CASE_SPEC["ct_name"], CASE_SPEC["gt_name"]]
                    if sorted(archive.namelist()) != sorted(names):
                        raise ValueError("Input ZIP must contain exactly the two expected NIfTI files")
                    for name, key in zip(names, ("ct", "gt")):
                        if archive.getinfo(name).file_size != CASE_SPEC["file_sizes"][key]:
                            raise ValueError("Input ZIP member size does not match the expected case")
                        with archive.open(name) as source, (INPUT_DIR / name).open("wb") as destination:
                            shutil.copyfileobj(source, destination)
        if not inputs_ok(INPUT_DIR):
            raise ValueError("CT/mask are missing or do not match the expected A-0025 bytes")
        cache.write_text(json.dumps({"directory": str(INPUT_DIR)}))
    except Exception as error:
        UPLOAD_ERROR = str(error)
        print("Upload check failed:", UPLOAD_ERROR)
else:
    print("Reusing verified uploaded CT/GT.")
CT_PATH, MASK_PATH = INPUT_DIR / CASE_SPEC["ct_name"], INPUT_DIR / CASE_SPEC["gt_name"]
print("Input folder:", INPUT_DIR)

## 5. Run the native real-data pilot
The helper verifies source hashes, TRAIN membership, native geometry and pixels
before allocating the large model. It runs the existing model/loss/AdamW/AMP
policy with checkpointed execution for the two fixed central masters, chosen
without using lesion labels. All requested optimizer updates must succeed.
It records actual losses, memory, timings, parameter changes, source-file
integrity and one CT-only master prediction. The data/loss are not synthetic.
This bounded pilot uses the shared Dataset/model/loss directly, not full Trainer
fit/resume, which still requires the complete cohort manifest and source files.

In [ ]:
destination = RUN / "real_case.json"
if UPLOAD_ERROR:
    report = {"status": "FAIL", "stage": "upload", "error": UPLOAD_ERROR, "optimizer_steps": 0}
    destination.write_text(json.dumps(report, indent=2))
else:
    environment = os.environ.copy()
    environment["PYTHONPATH"] = str(PROJECT / "src") + os.pathsep + environment.get("PYTHONPATH", "")
    command = [sys.executable, str(CANDIDATE_FILE), "--config", str(PROJECT / "configs/merit_index.yaml"),
        "--ct", str(CT_PATH), "--mask", str(MASK_PATH),
        "--split", str(PROJECT / "splits/covid19_20_5fold.json"), "--spec", str(SPEC_FILE),
        "--output", str(destination), "--steps", str(TRAIN_STEPS),
        "--memory-fraction", str(CUDA_MEMORY_FRACTION)]
    print("Starting the real CT pilot. No full epoch or accuracy evaluation is requested.", flush=True)
    with (RUN / "real_case.log").open("w") as log:
        process = subprocess.Popen(command, cwd=PROJECT, env=environment, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        exit_code = process.wait()
    report = json.loads(destination.read_text()) if destination.exists() else {
        "status": "FAIL", "error": "Worker produced no report; inspect real_case.log", "optimizer_steps": 0}
    report["worker_exit_code"] = exit_code
summary = {key: report.get(key) for key in ("status", "case_id", "optimizer_steps", "amp_skipped_steps",
    "weights_changed", "source_files_unchanged", "input_shape", "peak_allocated_mib", "peak_reserved_mib",
    "batches", "inference", "error")}
(RUN / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
print(json.dumps(summary, indent=2))

In [ ]:
BASELINE = {
    "source_report": "20261113_081310_7f8b19a4.zip",
    "mode": "previous_all_region_trial", "hardware": "Tesla T4", "torch": "2.14.1+cu130",
    "optimizer_steps": 2, "step_seconds": [13.958123673999978, 13.730988515999968],
    "peak_allocated_mib": 4649.67919921875, "peak_reserved_mib": 5072.0,
    "parameter_count": 174043602,
    "initial_parameter_sha256": "d323ad079ade4ad3f915190b59a59e37e02b900fab35f3c2211a6b7bdc79358a",
}
candidate_steps = [row["seconds"] for row in report.get("batches", []) if not row["skipped"]]
comparison = {
    "scope": "one case/two updates; descriptive comparison only",
    "baseline": BASELINE,
    "candidate": {key: report.get(key) for key in ("execution_mode", "status", "optimizer_steps",
        "peak_allocated_mib", "peak_reserved_mib", "parameter_count", "error")},
    "same_hardware_and_torch": report.get("hardware") == BASELINE["hardware"] and ENVIRONMENT["torch"] == BASELINE["torch"],
    "same_initial_parameters": report.get("parameter_sha256_before") == BASELINE["initial_parameter_sha256"],
    "candidate_step_seconds": candidate_steps,
}
if report.get("status") == "PASS" and len(candidate_steps) == 2:
    baseline_mean = sum(BASELINE["step_seconds"]) / 2
    candidate_mean = sum(candidate_steps) / 2
    comparison.update(baseline_mean_seconds=baseline_mean, candidate_mean_seconds=candidate_mean)
    if comparison["same_hardware_and_torch"] and comparison["same_initial_parameters"]:
        comparison["observed_time_change_percent"] = 100 * (candidate_mean / baseline_mean - 1)
        print("Negative time change means faster in this short sample; it is not a reliable cohort speedup.")
    else:
        comparison["time_change_not_computed"] = "Hardware/Torch/initial parameters differ from the reference."
(RUN / "comparison.json").write_text(json.dumps(comparison, indent=2) + "\n")
print(json.dumps(comparison, indent=2))

## 6. Download the small report ZIP
The report contains logs, specifications, helper hashes, loss/timing/memory and
integrity results. It excludes CT/GT input files and large model weights.
Use the download cell, or Files > Refresh > merit-results > ZIP menu > Download.
The printed `/content/...` path is a filesystem path, not a browser download URL.

In [ ]:
import shutil
archive = Path(shutil.make_archive(str(RUN), "zip", root_dir=RUN))
print("Report ZIP:", archive)
print("Download using the next cell, or the Files panel > ZIP menu > Download.")

In [ ]:
# On Colab this downloads the completed report. On Kaggle use the Output files.
try:
    import google.colab
except ImportError:
    print("Open the report ZIP from the notebook's output files:", archive)
else:
    from google.colab import files
    files.download(str(archive))
    print("If no download appears, open Files on the left, refresh, then merit-results > ZIP menu > Download.")